[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sampriti-sr/MLP_Course/blob/kshitij-variant/docs/recsys/09-two-stage.ipynb)

# Two-Stage Recommenders: Retrieval, then Ranking

<div class="admonition info">
<p class="admonition-title">What you will be able to do after this page</p>
<ul>
<li>Explain why production recommenders retrieve a few hundred candidates cheaply and then rank only those.</li>
<li>Measure how much of the future a retrieval stage can reach, and the ceiling that sets for any ranker.</li>
<li>Train a ranker on a <em>later</em> window than the one its features come from, and say why that order matters.</li>
<li>Judge a ranker per user, not by a pooled AUC, and decide whether it earns its complexity.</li>
</ul>
</div>

**Prerequisites:** [Matrix factorisation](05-factorisation.ipynb) and [Beyond accuracy](08-beyond-accuracy.ipynb). **Dataset:** MovieLens small. **Time:** about 45 minutes.

## The situation

So far every model scored *every* movie for *every* user and took the top ten. That is fine for 9,724 movies. A real catalogue has millions of items, a request must be answered in tens of milliseconds, and there is a lot more to know about a (user, item) pair than one score: the user's history, the item's age, how it was shown, the time of day. The standard answer is **two stages**: a fast **retrieval** stage that cuts the catalogue to a few hundred **candidates**, and a slower, richer **ranker** that orders only those. This page builds both on MovieLens, and tests whether the second stage earns its keep.

In [1]:
try:
    import mlp_data
except ImportError:  # e.g. a fresh Colab runtime
    %pip install -q "mlp-course @ git+https://github.com/sampriti-sr/MLP_Course@kshitij-variant"
    import mlp_data

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from mlp_data import load_movielens, recsys as rs, style

style.apply()
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:,.4f}".format)

## Maths ↔ code

- **Retrieval:** for each user, the $N$ best-scoring unseen items under a cheap model (here, the implicit ALS of page 5, $N = 100$). Its quality is **recall@N**: the share of the user's future hits that are among the candidates.
- **Ranker:** a classifier $\hat p(\text{hit} \mid u, i, x_{ui})$ on features $x_{ui}$ of the pair, trained with gradient-boosted trees (`HistGradientBoostingClassifier`). The final list is the 10 candidates with the highest $\hat p$.
- **Ceiling:** a ranker can only reorder the candidates, so its best possible NDCG@10 is that of an *oracle* that puts the candidates that are hits first.
- **Cost:** a single-stage model scores $U \times I$ pairs; the two-stage system scores $U \times N$ with the ranker.

## 0. Three windows, in time order

The ranker needs labelled examples, and the labels must come from a time **after** the history its features are computed from. So the data is used in three nested windows:

| | History (features, retrieval) | Labels (what the ranker learns to predict) | Used for |
|---|---|---|---|
| **Training the ranker** | the first 64% of each user's ratings | the next 16% | fit the ranker |
| **Final test** | the first 80% | the last 20% | report |

In [2]:
ratings = load_movielens().ratings
movies = load_movielens().movies

task = rs.make_task(ratings)                                   # history = first 80%, test = last 20%
inner = rs.make_task(task.train, index=task.index)             # history = first 64%, labels = the next 16%
n_users, n_items = task.X.shape
N = 100                                                        # candidates per user

genres, genre_names = rs.genre_matrix(movies, task.index)
genre_dense = genres.toarray().astype(np.float32)
genre_norm = np.maximum(np.linalg.norm(genre_dense, axis=1), 1e-9)
year = movies.set_index("movieId")["title"].str.extract(r"\((\d{4})\)\s*$")[0].astype(float).reindex(task.index.items).to_numpy()
year = np.where(np.isnan(year), np.nanmedian(year), year)

def likes(X):
    L = X.copy(); L.data = (L.data >= 4).astype(np.float32); L.eliminate_zeros(); return L

print(f"ranker training window: history {len(inner.train):,} ratings -> labels {len(inner.test):,} ratings")
print(f"final test window:      history {len(task.train):,} ratings -> test   {len(task.test):,} ratings")

ranker training window: history 64,549 ratings -> labels 16,123 ratings
final test window:      history 80,672 ratings -> test   20,164 ratings


## 1. Stage one: retrieval

Retrieval uses the tuned liked-only ALS model. First, how good is it as a *filter*? Recall@N is the share of a user's hits that make it into the top $N$:

In [3]:
def retrieval_model(t):
    L = likes(t.X)
    als = rs.als_implicit(L, factors=32, reg=10, alpha=1, iters=10, seed=0).scores()
    knn = rs.item_knn_scores(L, rs.item_similarity(L, k=50, shrink=10))
    return L, als, knn

L_test, als_test, knn_test = retrieval_model(task)

def recall_at(scores, t, n):
    top = rs.top_k(scores, t.B, n)
    found = np.take_along_axis(t.relevant.toarray() > 0, top, axis=1).sum(axis=1)
    total = np.asarray(t.relevant.sum(axis=1)).ravel()
    return (found[total > 0] / total[total > 0]).mean()

recall = pd.Series({n: recall_at(als_test, task, n) for n in (10, 50, 100, 200, 500, 1000)}, name="recall@N of the ALS retrieval")
assert 0.3 < recall[100] < 0.45 and recall[1000] > recall[100] + 0.2           # guards the reading below
recall.to_frame().T.rename_axis(columns="N")

N,10,50,100,200,500,1000
recall@N of the ALS retrieval,0.0945,0.2726,0.3837,0.5050,0.6772,0.7661


The top ten holds 9.5% of a user's future hits (that is the one-stage list); the top 100 holds **38%**, the top 200 half, the top 500 68% and the top 1,000 77%. Recall keeps climbing instead of plateauing, so $N = 100$ is a choice about cost, not a natural cut-off: with 100 candidates the ranker never even sees 62% of the hits.

## 2. Stage two: features and a ranker

For every (user, candidate) pair, the ranker sees the retrieval scores *and* things the retrieval model never used: how popular and how well liked the movie is, how active and how generous the user is, how well the movie's genres match the user's liked genres, and how far the movie's release year is from the years of the movies the user liked.

In [4]:
def candidate_features(t):
    L, als, knn = retrieval_model(t)
    z = lambda S: (S - S.mean(axis=1, keepdims=True)) / np.maximum(S.std(axis=1, keepdims=True), 1e-9)
    zals, zknn = z(als), z(knn)
    candidates = rs.top_k(als, t.B, N)                                           # stage one
    u, i = np.repeat(np.arange(n_users), N), candidates.ravel()

    counts = np.asarray(t.B.sum(axis=0)).ravel()
    liked_counts = np.asarray(L.sum(axis=0)).ravel()
    user_n = np.asarray(t.B.sum(axis=1)).ravel()
    user_liked = np.asarray(L.sum(axis=1)).ravel()
    profile = (L @ genres).toarray() / np.maximum(user_liked, 1)[:, None]        # mean genre vector of liked movies
    liked_year = (L @ year) / np.maximum(user_liked, 1)

    frame = pd.DataFrame({
        "user": u, "item": i, "rank": np.tile(np.arange(N), n_users),
        "als_z": zals[u, i], "knn_z": zknn[u, i],
        "log_item_popularity": np.log1p(counts[i]), "item_like_rate": (liked_counts[i] + 1) / (counts[i] + 2),
        "item_mean_rating": rs.damped_mean(t.X, 10)[i],
        "log_user_ratings": np.log1p(user_n[u]), "user_mean_rating": (np.asarray(t.X.sum(axis=1)).ravel() / np.maximum(user_n, 1))[u],
        "user_like_share": (user_liked / np.maximum(user_n, 1))[u],
        "genre_match": (profile[u] * genre_dense[i]).sum(axis=1) / (np.maximum(np.linalg.norm(profile, axis=1), 1e-9)[u] * genre_norm[i]),
        "year_gap": year[i] - liked_year[u], "item_year": year[i],
    })
    return frame, candidates

start = time.perf_counter()
train_frame, train_candidates = candidate_features(inner)          # features from the first 64%
test_frame, test_candidates = candidate_features(task)             # features from the first 80%
train_label = np.asarray(inner.relevant[train_frame["user"], train_frame["item"]]).ravel()     # hits in the NEXT 16%
test_label = np.asarray(task.relevant[test_frame["user"], test_frame["item"]]).ravel()         # hits in the last 20%
features = [c for c in train_frame.columns if c not in ("user", "item")]
print(f"{len(train_frame):,} training pairs, {len(test_frame):,} test pairs, {len(features)} features, built in {time.perf_counter() - start:.0f} s")
print(f"share of candidates that are hits: {train_label.mean():.1%} in the training window, {test_label.mean():.1%} in the test window")

61,000 training pairs, 61,000 test pairs, 12 features, built in 4 s
share of candidates that are hits: 3.8% in the training window, 4.2% in the test window


In [5]:
def fit_ranker(columns):
    ranker = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=100,
                                            l2_regularization=1.0, random_state=0)
    return ranker.fit(train_frame[columns], train_label)

def final_lists(ranker, columns):
    p = ranker.predict_proba(test_frame[columns])[:, 1].reshape(n_users, N)
    order = np.argsort(-p, axis=1, kind="stable")[:, :10]
    return np.take_along_axis(test_candidates, order, axis=1)

def ndcg_of(top):
    return rs.rank_metrics(top, task.relevant)["ndcg"]

stage_one = ndcg_of(test_candidates[:, :10])                                     # just the first ten candidates: the one-stage ALS list
oracle_order = np.argsort(-test_label.reshape(n_users, N), axis=1, kind="stable")[:, :10]
oracle = ndcg_of(np.take_along_axis(test_candidates, oracle_order, axis=1))      # hits first: the best any reordering could do

variants = {
    "ALS retrieval only (no ranker)": stage_one,
    "ranker, all features": ndcg_of(final_lists(fit_ranker(features), features)),
    "ranker, without the retrieval scores": ndcg_of(final_lists(fit_ranker([c for c in features if c not in ("als_z", "knn_z", "rank")]),
                                                                [c for c in features if c not in ("als_z", "knn_z", "rank")])),
    "ranker, retrieval scores only": ndcg_of(final_lists(fit_ranker(["als_z", "knn_z", "rank"]), ["als_z", "knn_z", "rank"])),
    "oracle reordering (ceiling)": oracle,
}
rows = {}
for name, s in variants.items():
    d = rs.paired_difference(s, stage_one)
    rows[name] = {"test NDCG@10": s.mean(), "vs retrieval only": "" if s is stage_one else f"{d['diff']:+.4f} [{d['lo']:+.4f}, {d['hi']:+.4f}]"}
result = pd.DataFrame(rows).T
d_all = rs.paired_difference(variants["ranker, all features"], stage_one)
d_no_cf = rs.paired_difference(variants["ranker, without the retrieval scores"], stage_one)
assert d_all["lo"] < 0 < d_all["hi"] and d_no_cf["hi"] < 0 and oracle.mean() > 3 * stage_one.mean()          # guards the reading below
result

,test NDCG@10,vs retrieval only
ALS retrieval only (no ranker),0.1209,
"ranker, all features",0.1202,"-0.0006 [-0.0078, +0.0066]"
"ranker, without the retrieval scores",0.0956,"-0.0253 [-0.0372, -0.0133]"
"ranker, retrieval scores only",0.1167,"-0.0042 [-0.0102, +0.0018]"
oracle reordering (ceiling),0.5957,"+0.4748 [+0.4525, +0.4971]"


Read the table from the top (every difference is paired against the retrieval-only list).

- **The ranker with all features ties retrieval alone**: 0.1202 against 0.1209, difference -0.0006 with an interval of [-0.008, +0.007]. On this data the second stage added nothing measurable.
- **Without the retrieval scores** it is clearly worse (0.0956, interval [-0.037, -0.013]): popularity, genre, year and user features cannot do the job of the collaborative score.
- **With the retrieval scores only** it also ties (0.1167): the extra features did not reliably help.
- The **oracle** row is the point of the ceiling: if the hits inside the 100 candidates were put first, NDCG@10 would be 0.596, five times the current value. There is a great deal of room *in principle* for a ranker; these features just do not locate it. The information about which of a user's 100 candidates they will like is not in the 12 features.

## 3. What goes wrong (1): a pooled AUC flatters the ranker

The usual way to report a classifier is its AUC. Compute it for the ranker, and for the single feature `als_z` that already ranked the candidates:

In [6]:
ranker = fit_ranker(features)
pooled_ranker = roc_auc_score(test_label, ranker.predict_proba(test_frame[features])[:, 1])
pooled_als = roc_auc_score(test_label, test_frame["als_z"])

def mean_per_user_auc(scores):                                        # AUC within each user's 100 candidates, then averaged
    s, y = scores.reshape(n_users, N), test_label.reshape(n_users, N)
    ok = (y.sum(axis=1) > 0) & (y.sum(axis=1) < N)
    return np.mean([roc_auc_score(y[u], s[u]) for u in np.flatnonzero(ok)])

per_user_ranker = mean_per_user_auc(ranker.predict_proba(test_frame[features])[:, 1])
per_user_als = mean_per_user_auc(test_frame["als_z"].to_numpy())
auc_table = pd.DataFrame({"pooled AUC (all pairs together)": [pooled_ranker, pooled_als], "mean per-user AUC": [per_user_ranker, per_user_als]},
                         index=["ranker, all features", "ALS score alone"])
assert pooled_ranker - pooled_als > 0.15 and abs(per_user_ranker - per_user_als) < 0.03          # guards the reading below
auc_table

,pooled AUC (all pairs together),mean per-user AUC
"ranker, all features",0.7298,0.6292
ALS score alone,0.5164,0.6196


Where does the pooled AUC come from? Shuffle one feature at a time and see how much the pooled AUC drops (permutation importance):

In [7]:
from sklearn.inspection import permutation_importance
imp = permutation_importance(ranker, test_frame[features], test_label, scoring="roc_auc", n_repeats=3, random_state=0)
importance = pd.Series(imp.importances_mean, index=features).sort_values(ascending=False)
user_level = {"user_like_share", "user_mean_rating", "log_user_ratings"}
print(f"features that describe only the user (the same for all 100 of that user's candidates): "
      f"{importance[list(user_level)].sum() / importance.clip(lower=0).sum():.0%} of the total importance")
importance.head(6).to_frame("drop in pooled AUC when shuffled")

features that describe only the user (the same for all 100 of that user's candidates): 67% of the total importance


,drop in pooled AUC when shuffled
log_user_ratings,0.1423
rank,0.0367
knn_z,0.0190
genre_match,0.0076
user_mean_rating,0.0076
user_like_share,0.0064


By the pooled AUC the ranker looks far better than the ALS score (**0.73 against 0.52**). Within each user's own list the two are almost the same (**0.629 against 0.620**), and the NDCG table above shows a tie.

**67% of the importance sits in features that describe only the user** (the same value for all 100 of that user's candidates), led by the number of ratings the user has. They cannot reorder one user's list, but they help tell *which users have many hits among their candidates*, which is exactly what a pooled AUC rewards. The ranker has learned "heavy users have more hits" and has been credited for ordering. The product shows each user one list, so judge a ranker by per-user metrics.

## 4. What goes wrong (2): fitting the ranker on the answers

The order of the windows is the whole point. What happens if the ranker is trained on the *test* labels, as can happen by accident when features and labels are built from one split?

In [8]:
leaky = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=100,
                                       l2_regularization=1.0, random_state=0).fit(test_frame[features], test_label)
p = leaky.predict_proba(test_frame[features])[:, 1].reshape(n_users, N)
leaky_top = np.take_along_axis(test_candidates, np.argsort(-p, axis=1, kind="stable")[:, :10], axis=1)
pd.Series({"honest ranker (trained on the earlier window)": variants["ranker, all features"].mean(),
           "leaky ranker (trained on the test labels)": ndcg_of(leaky_top).mean(),
           "oracle ceiling": oracle.mean()}, name="test NDCG@10").to_frame()
assert ndcg_of(leaky_top).mean() > variants["ranker, all features"].mean() * 1.3

Trained on the test labels, the same ranker reports **0.1894 against the honest 0.1202**, a 58% "improvement" that is pure leakage: the model has been shown the answers and memorises them. The honest number is the one that matches what happens in production, where the features describe the past and the labels are the unseen future.

## 5. What it buys, and what it costs

In [9]:
full = n_users * n_items
staged = n_users * N
print(f"pairs scored by a single-stage model: {full:,}")
print(f"pairs scored by the ranker in a two-stage system: {staged:,} ({staged / full:.1%} of that)")
catalogue = 5_000_000                                                  # a catalogue of a size where scoring everything per request is not an option
print(f"at {catalogue:,} items, one user's request would need {catalogue:,} scores single-stage against {N} for the ranker")

pairs scored by a single-stage model: 5,931,640
pairs scored by the ranker in a two-stage system: 61,000 (1.0% of that)
at 5,000,000 items, one user's request would need 5,000,000 scores single-stage against 100 for the ranker


The ranker scores **1.0%** of the (user, item) pairs a single-stage model would. At a catalogue of five million items the contrast is 5,000,000 scores against 100 for each request, which is why the structure is universal at scale. On this catalogue the saving is not needed and, as section 2 showed, the second stage adds no accuracy: the two-stage structure is not *justified* here. It is justified by cost at scale and by features the first stage cannot use.

<div class="admonition warning">
<p class="admonition-title">Watch out</p>
<p>A second stage adds a model to train, features to keep in sync between training and serving, and a time-ordered data set to maintain. If it does not beat the first stage by an amount you can measure, it is cost without benefit. Compare against the one-stage baseline, per user, with an interval.</p>
</div>

## Decision guide

| Situation | Use | Why |
|---|---|---|
| Small catalogue (thousands), one good score | Single-stage | Scoring everything is cheap and a ranker adds nothing measurable |
| Millions of items, tight latency | Retrieval (ALS, two-tower, ANN index) then a ranker | Only a few hundred pairs can afford rich features per request |
| Many signals the retrieval model cannot use (context, freshness, price) | A ranker | Features are where a ranker can add information |
| Choosing $N$ | Recall@N curve: the smallest $N$ whose recall is near the plateau | The ranker cannot recover hits the retrieval stage dropped |
| Evaluating | Per-user ranking metrics on a later window, with a paired interval | Pooled AUC and in-sample scores both mislead |

## Check yourself

<details class="question">
<summary>1. The retrieval stage keeps 100 candidates and finds 38% of a user's future hits. What is the most the ranker can find?</summary>

38% of them: the ranker only reorders what retrieval kept, so retrieval recall is a hard ceiling. The oracle row above is that ceiling expressed in NDCG. Raising the ranker's quality cannot help a hit that never reached it; raising $N$ or improving retrieval can.

</details>

<details class="question">
<summary>2. Why must the ranker's training labels come from a later window than the history its features are computed from?</summary>

At serving time the features describe the past and the label is in the future. A ranker trained with features and labels from the same moment learns patterns that do not exist at serving time (for example, "this item was in the user's history" would be a perfect predictor of a hit if the history included the answer). Section 4 shows how large the resulting over-optimism is.

</details>

## Exercises

<details class="question">
<summary>Exercise 1 (easy): how many candidates are needed for the retrieval stage to find half of the users' hits?</summary>

```python
print(recall)
```

About **200 candidates**: recall is 0.384 at 100 and 0.505 at 200. A larger $N$ raises the ceiling but costs the ranker more work per request, and the curve is still climbing at 1,000.

</details>

<details class="question">
<summary>Exercise 2 (medium): which features does the ranker rely on? Permutation importance on the test candidates answers it.</summary>

```python
from sklearn.inspection import permutation_importance
imp = permutation_importance(ranker, test_frame[features], test_label, scoring="roc_auc", n_repeats=3, random_state=0)
print(pd.Series(imp.importances_mean, index=features).sort_values(ascending=False))
```

The cell in section 3 already does this with the pooled AUC: user-only features take 67% of the credit. A feature that is constant within a user's candidates cannot reorder that user's list on its own, so an importance computed on the *per-user* AUC would credit them far less. Try it.

</details>

<details class="question">
<summary>Exercise 3 (hard): add a feature that the retrieval model could not use, such as the average rating of the movie in the *last year* of the history. Does the paired interval against retrieval-only move?</summary>

Build it from `inner.train` and `task.train`, compute it with a time filter, and add it to `candidate_features`. Decide in advance what size of gain you would call worth the extra feature.

</details>

## Recap

- **Two stages** exist for cost: retrieve a few hundred candidates cheaply, rank only those with rich features.
- Retrieval **recall@N is a ceiling** on the whole system; an oracle that reorders the candidates shows how much room the ranker has.
- Train the ranker on a **later window** than its features; otherwise its test score is meaningless.
- Judge it **per user, with a paired interval**. A pooled AUC rewards telling users apart, not ordering one user's list.
- On this small catalogue the second stage does not beat a good first stage; its case is cost at scale and signals the first stage cannot use.